# Setup for Debugging Tutor Notebooks

### Using HuggingFace Hub + llama.cpp

This notebook downloads the three GGUF models used in the Debugging Tutor Demo and checks whether the environment is ready (CPU-only or ~4GB VRAM settings).

### What is the Debugging Tutor?
The demo is a AI assistant that helps students debug by guiding them through **Diagnosis → Root Cause → Check → Review**, instead of giving full solutions.

### Why use three small local models and a large API model?
We use three Qwen2.5 1.5B to show that model behavior depends not only on the prompt and context, but also on how the model was trained. We also use a large API model for higher performance (optional LLM-as-Judge), while demonstrating cost and infrastructure tradeoffs. This comparison helps illustrate why model choice, evaluation, and training matter in AI workflows.

### Key experiments of the Debugging Tutor Demo

- **Prompting**: Pedagogical guardrails + Few-shot
- **Context scaling**: Progressively add context
- **Gradio**: Chatbot 
- **Evaluation(optional)**: LLM-as-Judge

## 0. Models
- Model Series: Qwen2.5 
- Parameters: 1.54B
- Q4 (4-bit) quantization for CPU-friendly
- 128K context window, 8K max generation.


| Model | Training Stage | Data | Notes |
| --- | --- | --- | --- |
| **Qwen2.5-1.5B** | Pretraining | General model trained on ~18T mixed tokens (text + some code) | Base language model |
| **Qwen2.5-Coder-1.5B** | Pretraining | Code-specialized model trained on ~5.5T code-focused tokens (source code, text-code grounding, synthetic data) | Significantly improvements in code generation, code reasoning and code fixing |
| **Qwen2.5-Coder-1.5B-Instruct** | Pretraining + Post-training | Coder + Instruction dataset | Coder + Significant improvements in instruction following, generating long texts, and generating structured outputs |


Model Cards: [Qwen2.5-1.5B](https://huggingface.co/QuantFactory/Qwen2.5-1.5B-GGUF), [Qwen2.5-Coder-1.5B](https://huggingface.co/QuantFactory/Qwen2.5-Coder-1.5B-GGUF), [Qwen2.5-Coder-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF).

References: `LlamaCpp_SmallLM_Demo.ipynb`, `HuggingFace_Hub_Download_gguf.ipynb`

## 1. Setup

Install and import HuggingFace Hub.

In [12]:
# Install/import HuggingFace Hub for model downloads.
try:
    from huggingface_hub import hf_hub_download, get_hf_file_metadata, hf_hub_url
except Exception:
    %pip install -q -U huggingface_hub
    from huggingface_hub import hf_hub_download, get_hf_file_metadata, hf_hub_url

from pathlib import Path
import os, time, shutil, subprocess


### 1.1 Pick your environment - Local vs Hub - and set the Path


In [13]:
# Hub path (DataHub / JupyterHub)
TARGET_DIR = Path('/home/jovyan/shared/')

# Local path example (if you run outside DataHub)
# TARGET_DIR = Path('./shared-rw').resolve()

TARGET_DIR.mkdir(parents=True, exist_ok=True)

# repo_id + filename: GGUF for llama.cpp & 4-bit quantized models (Q4) for CPU-only or ~4GB GPU.
# https://huggingface.co/repo_id/blob/main/filename
MODELS = [
    {
        # https://huggingface.co/QuantFactory/Qwen2.5-1.5B-GGUF/blob/main/Qwen2.5-1.5B.Q4_K_M.gguf
        'repo_id': 'QuantFactory/Qwen2.5-1.5B-GGUF',
        'filename': 'Qwen2.5-1.5B.Q4_K_M.gguf',
    },
    {
        # https://huggingface.co/QuantFactory/Qwen2.5-Coder-1.5B-GGUF/blob/main/Qwen2.5-Coder-1.5B.Q4_K_M.gguf
        'repo_id': 'QuantFactory/Qwen2.5-Coder-1.5B-GGUF',
        'filename': 'Qwen2.5-Coder-1.5B.Q4_K_M.gguf',
    },
    {
        # https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/blob/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf
        'repo_id': 'Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF',
        'filename': 'qwen2.5-coder-1.5b-instruct-q4_k_m.gguf',
    },
]

print('TARGET_DIR:', TARGET_DIR)
for m in MODELS:
    print(f"- {m['repo_id']} | {m['filename']}")


TARGET_DIR: /home/jovyan/shared
- QuantFactory/Qwen2.5-1.5B-GGUF | Qwen2.5-1.5B.Q4_K_M.gguf
- QuantFactory/Qwen2.5-Coder-1.5B-GGUF | Qwen2.5-Coder-1.5B.Q4_K_M.gguf
- Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF | qwen2.5-coder-1.5b-instruct-q4_k_m.gguf


## 2. Check Environment

Check model size, free disk space, and CPU-only or ~4GB GPU runtime settings.
Then copy the printed `DEFAULT_N_CTX` and `DEFAULT_N_GPU_LAYERS` into `Debugging_Tutor_Demo.ipynb`.


In [14]:
def total_ram_gb() -> float:
    """Return total system RAM in GB (or -1 if unavailable)."""
    try:
        pages = os.sysconf('SC_PHYS_PAGES')
        page_size = os.sysconf('SC_PAGE_SIZE')
        return (pages * page_size) / (1024**3)
    except Exception:
        return -1.0


def max_vram_gb():
    """Return max NVIDIA VRAM in GB, or None if no GPU / nvidia-smi."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            stderr=subprocess.STDOUT,
            text=True,
        )
        vals = [float(x.strip()) / 1024 for x in out.strip().splitlines() if x.strip()]
        return max(vals) if vals else None
    except Exception:
        return None


# 1) Check model download size vs free disk in TARGET_DIR.
size_rows = []
total_bytes = 0
size_ok = True

for m in MODELS:
    try:
        url = hf_hub_url(repo_id=m['repo_id'], filename=m['filename'])
        meta = get_hf_file_metadata(url)
        total_bytes += meta.size
        size_rows.append((m['filename'], meta.size / (1024**3)))
    except Exception:
        size_ok = False
        size_rows.append((m['filename'], float('nan')))

total_gb = total_bytes / (1024**3)
free_gb = shutil.disk_usage(str(TARGET_DIR)).free / (1024**3)

print('--- Download size check ---')
for name, gb in size_rows:
    if gb == gb:  # NaN-safe check
        print(f'{name}: {gb:.2f} GB')
    else:
        print(f'{name}: size lookup failed')

if size_ok:
    print(f'Total expected download size: {total_gb:.2f} GB')
else:
    print('Total expected download size: unavailable')
print(f'Free disk in target path: {free_gb:.2f} GB')

if size_ok:
    # 20% margin helps avoid partial-download failures.
    disk_ok = free_gb >= total_gb * 1.2
    print('Disk status:', 'PASS' if disk_ok else 'WARN (free more disk space)')
else:
    print('Disk status: CHECK MANUALLY')

# 2) Suggest runtime defaults for CPU-only or ~4GB GPU.
cpu_cores = os.cpu_count() or 0
ram_gb = total_ram_gb()
vram_gb = max_vram_gb()

cpu_ready = (cpu_cores >= 4) and (ram_gb < 0 or ram_gb >= 8)
gpu_4gb_ready = (vram_gb is not None) and (vram_gb >= 4.0) and (ram_gb < 0 or ram_gb >= 8)

RECOMMENDED_N_CTX = 2048
RECOMMENDED_N_GPU_LAYERS = 20 if gpu_4gb_ready else 0

print('\n--- Runtime suggestion ---')
print('CPU cores:', cpu_cores)
print(f'Total RAM (GB): {ram_gb:.2f}' if ram_gb >= 0 else 'Total RAM (GB): unknown')
print('Max NVIDIA VRAM (GB):', f'{vram_gb:.2f}' if vram_gb is not None else 'not found')
print('CPU-only readiness:', 'PASS' if cpu_ready else 'WARN')
print('~4GB VRAM readiness:', 'PASS' if gpu_4gb_ready else 'WARN')

print('\nNext step: in Debugging_Tutor_Demo.ipynb')
print('DEFAULT_N_CTX =', RECOMMENDED_N_CTX)
print('DEFAULT_N_GPU_LAYERS =', RECOMMENDED_N_GPU_LAYERS)


--- Download size check ---
Qwen2.5-1.5B.Q4_K_M.gguf: 0.92 GB
Qwen2.5-Coder-1.5B.Q4_K_M.gguf: 0.92 GB
qwen2.5-coder-1.5b-instruct-q4_k_m.gguf: 1.04 GB
Total expected download size: 2.88 GB
Free disk in target path: 5.00 GB
Disk status: PASS

--- Runtime suggestion ---
CPU cores: 64
Total RAM (GB): 1133.60
Max NVIDIA VRAM (GB): 22.49
CPU-only readiness: PASS
~4GB VRAM readiness: PASS

Next step: in Debugging_Tutor_Demo.ipynb
DEFAULT_N_CTX = 2048
DEFAULT_N_GPU_LAYERS = 20


## 3. Download the model files

Each file is downloaded with retry logic for temporary network failures.


In [17]:
# Download each model with retry for temporary network errors.
def download_one(repo_id: str, filename: str, retries: int = 3):
    for attempt in range(1, retries + 1):
        try:
            return hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                local_dir=str(TARGET_DIR),
            )
        except Exception as e:
            print(f'Attempt {attempt}/{retries} failed for {filename}: {e}')
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)

for m in MODELS:
    print()
    print('Downloading:', m['filename'])
    saved = download_one(m['repo_id'], m['filename'])
    print('Saved:', saved)



Downloading: Qwen2.5-1.5B.Q4_K_M.gguf
Saved: /home/jovyan/shared/Qwen2.5-1.5B.Q4_K_M.gguf

Downloading: Qwen2.5-Coder-1.5B.Q4_K_M.gguf
Saved: /home/jovyan/shared/Qwen2.5-Coder-1.5B.Q4_K_M.gguf

Downloading: qwen2.5-coder-1.5b-instruct-q4_k_m.gguf
Saved: /home/jovyan/shared/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf


## 4. Verify downloaded files

Confirm that all target files exist in the selected directory.


In [16]:
# Verify each downloaded file exists and print its size.
missing = []
for m in MODELS:
    f = TARGET_DIR / m['filename']
    if not f.exists():
        missing.append(m['filename'])
    else:
        print(f'{f.name}: {f.stat().st_size / (1024**3):.2f} GB')

print('Missing files:', missing)
assert len(missing) == 0, 'Some model files are missing.'
print()
print('PASS: all listed model files are downloaded.')


Qwen2.5-1.5B.Q4_K_M.gguf: 0.92 GB
Qwen2.5-Coder-1.5B.Q4_K_M.gguf: 0.92 GB
qwen2.5-coder-1.5b-instruct-q4_k_m.gguf: 1.04 GB
Missing files: []

PASS: all listed model files are downloaded.


## Next Steps

- **Intro**: Demo + experiments
- **Intermediate**: Routing + RAG + metadata pipeline automation (optional metrics)
- **Advanced**: Multi-Agent System (RAG agnet + Exam agnet + Writing agent + Debugging agent)
- **Advanced(optional, GPU)**: Synthetic data pipeline + LoRA/distillation 